In [ ]:
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sunbird.inference.samples import Chain

from acm.utils.modules import get_class_from_module
from secondgen_bgs import get_secondgen_data

%config InlineBackend.figure_format='retina'

chain_dir_Mr20 = '/pscratch/sd/s/sbouchar/acm/bgs-20/chains/'
chain_dir_Mr21 = '/pscratch/sd/s/sbouchar/acm/bgs-21.35/chains/'


cosmo_params = ['omega_cdm', 'omega_b', 'sigma8_m', 'n_s', 'nrun', 'N_ur', 'w0_fld', 'wa_fld']
hod_params = ['logM_cut', 'logM_1', 'alpha', 'kappa', 'sigma', 'alpha_c', 'alpha_s', 's', 'A_cen', 'A_sat', 'B_cen', 'B_sat']

# default values
colors = ["#5ac3be", "#e770a2", "#4165c0", "#696969", "#f79a1e", "#ba7dcd"]

label_dict = {
    'tpcf': '2PCF', 
    'ds_xiqq+ds_xiqg': 'DS',
    'ds_xiqg+ds_xiqq': 'DS',
    'tpcf+ds_xiqq+ds_xiqg': '2PCF + DS',
    'tpcf+ds_xiqg+ds_xiqq': '2PCF + DS',
    'tpcf_s_1.00-150.00': '2PCF (1-150 Mpc/h)',
    'tpcf_s_30.00-150.00': '2PCF (30-150 Mpc/h)',
    'tpcf_s_1.00-150.00+ds_xiqq_s_1.00-30.00+ds_xiqg_s_1.00-30.00': '2PCF (1-150 Mpc/h) + DS (1-30 Mpc/h)',
    'tpcf_s_1.00-150.00+ds_xiqg_s_1.00-30.00+ds_xiqq_s_1.00-30.00': '2PCF (1-150 Mpc/h) + DS (1-30 Mpc/h)',
}

# For specific uses
colors2 = ["#182a88", "#105203", "#9ca6d8", "#8fde7f"]
label_dict2 = {
    "tpcf_s_1.00-150.00 (Mr20)": "2PCF (Mr < -20)",
    "tpcf_s_1.00-150.00+ds_xiqg_s_1.00-30.00+ds_xiqq_s_1.00-30.00 (Mr20)": "2PCF + DS (Mr < -20)",
    "tpcf_s_1.00-150.00 (Mr21)": "2PCF (Mr < -21)",
    "tpcf_s_1.00-150.00+ds_xiqq_s_1.00-30.00+ds_xiqg_s_1.00-30.00 (Mr21)": "2PCF + DS (Mr < -21)",
}

In [ ]:
def get_chain(
    type_fit: str, 
    cosmo_model: str, 
    hod_model: str, 
    stat_name: str, 
    identifier: str = None,
    chain_dir: Path|str = chain_dir_Mr20
) -> Chain:
    """Load a chain from disk trough the Chain class."""
    fn = stat_name
    if identifier is not None:
        fn += f'_{identifier}'
    chain_dir = Path(chain_dir)
    fn = chain_dir / type_fit / f'cosmo-{cosmo_model}_hod-{hod_model}' / f'{fn}.npy'
    if not fn.exists():
        return None
    chain = Chain.load(fn)
    chain.data['label'] = stat_name
    return chain

def get_chains(
    type_fit: str, 
    cosmo_model: str, 
    hod_model: str, 
    stats: list[str] = ['tpcf', 'ds_xiqg+ds_xiqq', 'tpcf+ds_xiqg+ds_xiqq'],
    identifier: str = None,
    chain_dir: Path|str = chain_dir_Mr20
) -> list[Chain]:
    """Get several chains corresponding to different statistics."""
    chains = []
    for stat in stats:
        chain = get_chain(type_fit, cosmo_model, hod_model, stat, identifier=identifier, chain_dir=chain_dir)
        if chain is not None:
            chains.append(chain)
    return chains

def print_std_improvements(ref: Chain, comp: Chain, params: list[str]) -> dict:
    """
    Print the standard deviation improvements from a reference chain to a comparison chain.
    
    Parameters
    ----------
    ref: Chain
        The reference chain.
    comp: Chain
        The comparison chain.
    params: list[str]
        The list of parameter names to consider.
    
    Returns
    -------
    improvements: dict
        A dictionary with parameter names as keys and a list of tuples (mean, std, improvement) as values.
    """
    
    improvements = {}
    
    chains = [ref, comp]
    chains_mean = [chain.samples.mean(axis=0) for chain in chains]
    chains_std = [chain.samples.std(axis=0) for chain in chains]
    names = chains[0].names
    
    for name in params:
        if name not in names:
            continue
        
        improvements.setdefault(name, [])
        for i, chain in enumerate(chains):
            mean = chains_mean[i][names.index(name)]
            std = chains_std[i][names.index(name)]
            
            if i == 0:
                ref_std = std
            else:
                improvement = (ref_std - std) / ref_std * 100
                improvements[name].append((mean, std, improvement))
                print(f'    {name}: {mean:.5f} ± {std:.5f} ({improvement:.2f}% std improvement)')
    
    return improvements

def add_model_errors_to_ax(
    observable,
    chain: Chain,
    ax: plt.Axes,
    **kwargs,
) -> None: 
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model_err = observable.get_emulator_error()
        mock_err = np.sqrt(np.diag(observable.get_covariance_matrix()))
        pred_err = np.sqrt(model_err**2 + mock_err**2)
        
    if observable.stat_name == 'tpcf':
        s = observable.s.values
        ells = kwargs.get('ells', [0, 2])
        for i, ell in enumerate(ells):
            y = observable.get_model_prediction(chain.bestfit).unstack().sel(ells=ell).values
            err = pred_err.unstack().sel(ells=ell).values
            ax[0].fill_between(s, (y - err)*s**2, (y + err)*s**2, alpha=0.2, color=f'C{i}', zorder=-1)
    elif observable.stat_name in ['ds_xiqq', 'ds_xiqg']:
        s = observable.s.values
        ell = kwargs.get('ell', 0)
        quantiles = kwargs.get('quantiles', [0, 1, 3, 4])
        for i, q in enumerate(quantiles):
            y = observable.get_model_prediction(chain.bestfit).sel(ells=ell, quantiles=q).values
            err = pred_err.sel(ells=ell, quantiles=q).values
            ax[0].fill_between(s, (y - err)*s**2, (y + err)*s**2, alpha=0.2, color=f'C{i}', zorder=-1)

def plot_acm_model_vs_truth(
    chain: Chain, 
    module: str = 'acm.observables.bgs', 
    observable_name: str = 'tpcf',
    obs_kwargs: dict = {},
    add_model_errors: bool = True,
    **kwargs,
) -> tuple[plt.Figure, plt.Axes, tuple]:
    """
    Use the plot_observable method from the BGS observables to plot 
    the validation mocks data vs the model prediction at the bestfit point of the chain.
    
    Parameters
    ----------
    chain : Chain
        The chain from which to get the bestfit parameters.
    module : str, optional
        The module where the observable class is located, by default 'acm.observables.bgs'
    observable_name : str, optional
        The name of the observable class to use the plot function from, by default 'tpcf'.
    add_model_errors : bool, optional
        Whether to add model errors to the plot, by default True.
        
    Returns
    -------
    fig : plt.Figure
        The figure object containing the plot.
    ax : plt.Axes
        The axes object containing the plot.
    tuple :
        A tuple containing the observable instance and the predicted errors.
    """
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        cls = get_class_from_module(module, observable_name)
        obs = cls(**obs_kwargs) 
        
        fig, ax = obs.plot_observable(chain.bestfit, show_legend=True)
        if add_model_errors:
            add_model_errors_to_ax(obs, chain, ax, **kwargs)
            
    return fig, ax



def plot_secondgen_model_vs_truth(
    chain: Chain, 
    module: str = 'acm.observables.bgs', 
    observable_name: str = 'tpcf',
    add_model_errors: bool = True,
    **kwargs,
) -> tuple[plt.Figure, plt.Axes, tuple]:
    """
    Use the plot_observable method from the BGS observables to plot 
    the secondgen data vs the model prediction at the bestfit point of the chain.
    
    Parameters
    ----------
    chain : Chain
        The chain from which to get the bestfit parameters.
    module : str, optional
        The module where the observable class is located, by default 'acm.observables.bgs'
    observable_name : str, optional
        The name of the observable class to use the plot function from, by default 'tpcf'.
    add_model_errors : bool, optional
        Whether to add model errors to the plot, by default True.
        
    Returns
    -------
    fig : plt.Figure
        The figure object containing the plot.
    ax : plt.Axes
        The axes object containing the plot.
    tuple :
        A tuple containing the observable instance and the predicted errors.
    """
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        obs = get_class_from_module(module, observable_name)()
        
        sg_obs = get_secondgen_data(obs, return_obs=True)
        obs._dataset = sg_obs[0]._dataset
        
        kwargs['volume_factor'] = 1.0 # Enforce no volume scaling for secondgen
        fig, ax = obs.plot_observable(chain.bestfit, show_legend=True, **kwargs)
        
        # Recreate obs instance for the errors
        obs = get_class_from_module(module, observable_name)()
    
    if add_model_errors:
        add_model_errors_to_ax(obs, chain, ax, **kwargs)
            
    return fig, ax

## Validation plots (c000 hod061)

### Free HOD

In [ ]:
chains = get_chains(type_fit='validation/c000_hod061', cosmo_model='None', hod_model='base-AB-CB-VB-s', identifier='bestfit')

markers = chains[0].markers
chains[0].plot_triangle(*chains[1:], add_bestfit=True, colors=colors, filled=True, title_limit=1, label_dict=label_dict, markers=markers, params=cosmo_params);

### LCDM

In [ ]:
chains = get_chains(type_fit='validation/c000_hod061', cosmo_model='base-fixed-omega_b', hod_model='base-AB-CB-VB-s', identifier='bestfit')

markers = chains[0].markers
chains[0].plot_triangle(*chains[1:], add_bestfit=True, colors=colors, filled=True, title_limit=1, label_dict=label_dict, markers=markers, params=cosmo_params);

fig = plt.gcf()
fig.suptitle(r'LCDM (fixed $\Omega_b$) fit to bestfit in validation data');

### w0wa

In [ ]:
chains = get_chains(type_fit='validation/c000_hod061', cosmo_model='base-w0wa', hod_model='base-AB-CB-VB-s', identifier='bestfit')

markers = chains[0].markers
chains[0].plot_triangle(*chains[1:], add_bestfit=True, colors=colors, filled=True, title_limit=1, label_dict=label_dict, markers=markers, params=cosmo_params);

fig = plt.gcf()
fig.suptitle('w0wa fit to bestfit in validation data');

## Plots pour c000 hod000

In [ ]:
chains = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_30.00-150.00', 'tpcf_s_1.00-150.00'],
    chain_dir=chain_dir_Mr21,
)

print('Standard deviation improvements for small-scales 2PCF (1-150 Mpc/h):')
print_std_improvements(chains[0], chains[1], cosmo_params)

markers = chains[0].markers
chains[0].plot_triangle(*chains[1:], add_bestfit=True, colors=colors, filled=True, title_limit=1, label_dict=label_dict, markers=markers, params=cosmo_params);

fig = plt.gcf()
fig.suptitle('w0wa fit to c000 validation mock (Mr<-21.35)');

In [ ]:
chains = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_1.00-150.00', 'tpcf_s_1.00-150.00+ds_xiqq_s_1.00-30.00+ds_xiqg_s_1.00-30.00'],
    chain_dir=chain_dir_Mr21,
)

print('Standard deviation improvements with DS (1-30 Mpc/h) added to 2PCF (1-150 Mpc/h):')
print_std_improvements(chains[0], chains[1], cosmo_params)

markers = chains[0].markers
chains[0].plot_triangle(*chains[1:], add_bestfit=True, colors=colors, filled=True, title_limit=1, label_dict=label_dict, markers=markers, params=cosmo_params);

fig = plt.gcf()
fig.suptitle('w0wa fit to c000 validation mock (Mr<-21.35)');

In [ ]:
chains_Mr21 = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_1.00-150.00', 'tpcf_s_1.00-150.00+ds_xiqq_s_1.00-30.00+ds_xiqg_s_1.00-30.00'],
    chain_dir=chain_dir_Mr21,
)

for chain in chains_Mr21:
    chain.data['label'] += ' (Mr21)'

chains_Mr20 = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_1.00-150.00', 'tpcf_s_1.00-150.00+ds_xiqg_s_1.00-30.00+ds_xiqq_s_1.00-30.00'],
    chain_dir=chain_dir_Mr20,
)

for chain in chains_Mr20:
    chain.data['label'] += ' (Mr20)'
    
print('Standard deviation improvements with Mr<-20 (2PCF):')
print_std_improvements(chains_Mr21[0], chains_Mr20[0], cosmo_params)

print('Standard deviation improvements with Mr<-20 (2PCF+DS):')
print_std_improvements(chains_Mr21[1], chains_Mr20[1], cosmo_params)

chains = chains_Mr21 + chains_Mr20

markers = chains[0].markers
chains[0].plot_triangle(
    *chains[1:], 
    add_bestfit=True, 
    colors=colors2, 
    filled=True, 
    title_limit=1, 
    label_dict=label_dict2, 
    markers=markers, 
    params=cosmo_params, 
    contour_args=dict(alpha=0.9)
);

fig = plt.gcf()
fig.suptitle('w0wa fit to c000 validation mock ');

In [ ]:
# Total improvement

chain_base = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_30.00-150.00'],
    chain_dir=chain_dir_Mr21,
)[0]

chain_final = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_1.00-150.00+ds_xiqg_s_1.00-30.00+ds_xiqq_s_1.00-30.00'],
    chain_dir=chain_dir_Mr20,
)[0]

print('Total standard deviation improvements (Mr<-20, 2PCF 1-150 Mpc/h + DS 1-30 Mpc/h):')
print_std_improvements(chain_base, chain_final, cosmo_params);

In [ ]:
# Model vs truth
label = chain_final.data['label']
stats = label.split('+')

Mr = -20
paths = dict(
    data_dir= f'/pscratch/sd/s/sbouchar/acm/bgs{Mr}/input_data/',
    model_dir= f'/pscratch/sd/s/sbouchar/acm/bgs{Mr}/trained_models/',
)

for stat in stats:
    stat_name = stat.split('_s')[0]
    fig, ax = plot_acm_model_vs_truth(chain_final, observable_name=stat_name, obs_kwargs=dict(paths=paths, select_filters={'cosmo_idx': 0, 'hod_idx': 0}))
    fig.suptitle(rf'w0wa (fixed $\Omega_b$) fit on validation mock - {label_dict[label]}');

## SecondGen plots

### LCDM

In [ ]:
chains = get_chains(
    type_fit='secondgen', 
    cosmo_model='base-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf', 'tpcf+ds_xiqg+ds_xiqq'],
)

print('Standard deviation improvements from validation mock to secondgen:')
print_std_improvements(chains[0], chains[1], cosmo_params)

markers = chains[0].markers
chains[0].plot_triangle(*chains[1:], add_bestfit=True, colors=colors, filled=True, title_limit=1, label_dict=label_dict, markers=markers, params=cosmo_params);

fig = plt.gcf()
fig.suptitle(r'LCDM (fixed $\Omega_b$) fit on SecondGen data');

### w0wa

In [ ]:
chains = get_chains(
    type_fit='secondgen', 
    cosmo_model='base-w0wa', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf', 'tpcf+ds_xiqg+ds_xiqq'],
)

print('Standard deviation improvements from validation mock to secondgen:')
print_std_improvements(chains[0], chains[1], cosmo_params)

markers = chains[0].markers
chains[0].plot_triangle(*chains[1:], add_bestfit=True, colors=colors, filled=True, title_limit=1, label_dict=label_dict, markers=markers, params=cosmo_params);

fig = plt.gcf()
fig.suptitle('w0wa fit on SecondGen data');

# Model prediction vs truth for SecondGen

In [ ]:
chains = get_chains(type_fit='secondgen', cosmo_model='base-fixed-omega_b', hod_model='base-AB-CB-VB-s')

for chain in chains:
    label = chain.data['label']
    stats = label.split('+')
    for stat in stats:
        fig, ax = plot_secondgen_model_vs_truth(chain, observable_name=stat)
        fig.suptitle(rf'LCDM (fixed $\Omega_b$) fit on SecondGen data - {label}');

## Old tests

### First plot testing the secondgen inference 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sunbird.inference.samples import Chain
from acm.observables.bgs import tpcf
from secondgen_bgs import get_secondgen_data

obs = tpcf(select_filters=dict(ells=0), numpy_output=True)
s = obs.s

sg_tpcf = get_secondgen_data(obs, return_obs=True)
truth = sg_tpcf.y
sg_err = np.sqrt(np.diag(sg_tpcf.get_covariance_matrix(volume_factor=1.0)))

pred = obs.get_model_prediction(sg_tpcf.x)

model_err = obs.get_emulator_error()
mock_err = np.sqrt(np.diag(obs.get_covariance_matrix()))
pred_err = np.sqrt(model_err**2 + mock_err**2)

chain = Chain.load('/pscratch/sd/s/sbouchar/acm/bgs-20/chains/secondgen/cosmo-base_hod-base-VB-AB-CB-s/tpcf.npy')

pred_fit = obs.get_model_prediction(chain.bestfit)

plt.errorbar(s, truth*s**2, yerr=sg_err*s**2, fmt='o', ms=3, label='SecondGen BGS')

plt.plot(s, pred*s**2, label='ACM Truth Prediction')
plt.fill_between(s, (pred - pred_err)*s**2, (pred + pred_err)*s**2, color='C1', alpha=0.3)

plt.plot(s, pred_fit*s**2, label='ACM Best-fit Prediction', linestyle='--')
plt.fill_between(s, (pred_fit - pred_err)*s**2, (pred_fit + pred_err)*s**2, color='C2', alpha=0.3)

plt.xlabel('s [Mpc/h]')
plt.ylabel(r'$s^2 \xi_0(s)$')
plt.title('BGS Monopole Correlation Function')
plt.legend();

### Plot examples

In [ ]:
# SecondGen LCDM w/ Full HOD
type_fit = 'secondgen'
cosmo_model = 'base'
hod_model = 'base-VB-AB-CB-s'

# Triange plot
chains = get_chains(type_fit, cosmo_model, hod_model)
markers = chains[0].markers
chains[0].plot_triangle(*chains[1:], add_bestfit=True, colors=colors, filled=True, title_limit=1, label_dict=label_dict, markers=markers, params=cosmo_params)

# MAP
chains[0].plot_map(*chains[1:], colors=colors, label_dict=label_dict, markers=markers, params=cosmo_params);

# Model vs Truth plot
fig, ax = plot_secondgen_model_vs_truth(chains[1], observable_name='ds_xiqq')